Python 3.11.9

In [ ]:
#!pip install pandas numpy matplotlib scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

**Cargo el fichero en formato excel:**

In [ ]:
#!pip install openpyxl

In [ ]:
#tengo el fichero en la misma carpeta
original_data = pd.read_excel('_NLP_litotricia_ID_estudio_original.xlsx')
patient_data = original_data.copy()
patient_data.head()

In [ ]:
N_pat, N_columns = patient_data.shape
print(f"Number of patients: {N_pat}")
print(f"Number of columns: {N_columns}")
patient_data.dtypes

# Primeros ajustes de formato

Cambio de nombres de columnas con error en codificacion:

In [ ]:
patient_data.columns.values.tolist()

In [ ]:
patient_data.rename(columns={"radiografÃƒÂ\xada": "radiografia"}, inplace=True)

In [ ]:
patient_data.columns.values.tolist()

Quito decimal sobrante de variables categoricas:

In [ ]:
# quitar decimal sobrante de variables categoricas
categorias = ['Sexo', 'Diabetes', 'HTA', 'Fumador_tabaco', 'ASA','radiografia','eco', 'UIV','cultivo','Cultresul','Microorganismo','TAC','urgencias',
              'motivo', 'tipotto', 'tecnica','lado', 'localizacion','Guy_Score','posicion_paciente','acceso_puncion',
              'cateterismo_ureteral','contraste','metodo_dilatacion','multitrayecto','fuente_fragmentacion',
              'facilidad_localizacion','anestesia','drenajes','tubeless','rxevaluacion','resultado_3meses',
              'Finalizado','Ttorescate','Tipottorescate','Complicaciones','tipocomplicacion','complicaciones_intra',
              'complicaciones_post','trasfusion','reintervcomplic','embolizacion','SANGRADO','PERFORACION',
              'PERDIDAACCESO','INFECCION','SEPSIS','HEMATURIA','HEMATOMA', 'Compli_Post_Recod','Complic_Intra_Recodif', 'IngresoSiNo']	
patient_data[categorias] = patient_data[categorias].astype('Int64')
patient_data.head()

Redondear el procentaje de neutrofilos:

In [ ]:
patient_data['ProcentajeNeutrofilos'] = patient_data['ProcentajeNeutrofilos'].round(decimals = 2)
patient_data.head()

# Identificación de errores y su eliminación

Reemplazar los errores en datos por un NaN en variables categoricas (Han sido identificados mediante analisis EDA disponible en SPSS)

In [ ]:
patient_data.loc[patient_data['acceso_puncion'] == 3, 'acceso_puncion'] = np.nan  
patient_data.loc[patient_data['PERFORACION'] == 67, 'PERFORACION'] = np.nan  
#comprobar:
patient_data.loc[patient_data['codigo'] == 'NLP387', 'acceso_puncion'].values


Eliminar el elemento marcado como duplicado en observaciones (DUPLICADO PORQUE LA PRIMERA VEZ NO SE REALIZÃ“)

In [ ]:
patient_data.drop(patient_data[patient_data['observaciones'].str.contains("DUPLICADO PORQUE LA PRIMERA VEZ NO SE REALIZÃ“", na=False)].index, inplace=True)
N_pat, N_columns = patient_data.shape
print(f"Number of patients: {N_pat}")

**Elección de las variables de interes**

In [ ]:
# Crear un nuevo DataFrame con columnas específicas
columnas_caract_antes = ['codigo', 'Sexo', 'Diabetes', 'HTA', 'Fumador_tabaco','BMI','ASA','Cultresul','Microorganismo','motivo', 'tipotto',
                         'lado','diametro_mayor','diametro_menor','numero', 'localizacion','Guy_Score','UH',
                         'distanciapielcalculo','EDAD']
columnas_quir=['tecnica', 'posicion_paciente','acceso_puncion',
              'cateterismo_ureteral','contraste','metodo_dilatacion','multitrayecto','calibre_vaina','fuente_fragmentacion',
              'facilidad_localizacion','duracion_tto','anestesia','drenajes','tubeless']
columnas_analitica=['Procalcitonina','LeucocitosTotales','ProcentajeNeutrofilos']
columnas_complicaciones = ['Complicaciones','tipocomplicacion','complicaciones_intra',
              'complicaciones_post','trasfusion','complicaclavien','reintervcomplic','embolizacion','SANGRADO','PERFORACION',
              'PERDIDAACCESO','INFECCION','SEPSIS','HEMATURIA','HEMATOMA']
etiqueta_a_predecir = ['HEMATURIA','HEMATOMA','SANGRADO','trasfusion','embolizacion']
#SELECCION DE COLUMNAS
selected_columns = columnas_caract_antes+columnas_quir+columnas_analitica+etiqueta_a_predecir
patient_data = patient_data[selected_columns]
patient_data.head()

Eliminar 6 pacientes con tumor, dado que tener una enfermedad grave se considera como factor de exclusion

In [ ]:
patient_data = patient_data[~patient_data['motivo'].isin([1, 2])]
N_pat, N_columns = patient_data.shape
print(f"Number of patients: {N_pat}")
N_columns

In [ ]:
patient_data.drop(columns=['motivo'],inplace=True)
patient_data.head()

**Umbralización de variables analíticas:**

In [ ]:
patient_data['Procalcitonina'] = np.where(
    patient_data['Procalcitonina'].notna(),
    np.where(patient_data['Procalcitonina'] >= 0.5, 1, 0),
    np.nan
)

patient_data['LeucocitosTotales'] = np.where(
    patient_data['LeucocitosTotales'].notna(),
    (
        (patient_data['LeucocitosTotales'] >= 15) |
        (patient_data['LeucocitosTotales'] <= 4.5)
    ).astype(int),
    np.nan
)

patient_data['ProcentajeNeutrofilos'] = np.where(
    patient_data['ProcentajeNeutrofilos'].notna(),
    (patient_data['ProcentajeNeutrofilos'] > 80).astype(int),
    np.nan
)

**Prospección de valores anómalos en variables númericas **

In [ ]:
numericas = ['BMI','diametro_mayor','diametro_menor','UH','distanciapielcalculo']+['numero','EDAD']+['calibre_vaina','duracion_tto']

In [ ]:
features = ['BMI','diametro_mayor','diametro_menor','numero','UH',
                         'distanciapielcalculo','EDAD']+['calibre_vaina','duracion_tto']
fig, axes = plt.subplots(nrows=len(features), ncols=3, figsize=(12, 3 * len(features)))
fig.suptitle("Distribuciones de caracteristicas numericas", fontsize=16)

plot_types = [
    ("Boxplot", lambda ax, data: ax.boxplot(data)),
    ("Histogram", lambda ax, data: ax.hist(data, bins=20, edgecolor='black')),
    ("Violin Plot", lambda ax, data: ax.violinplot(data))
]

for row_idx, feature in enumerate(features):
    column = patient_data[feature].dropna().values  # Convertimos en array 1D sin NaNs
    for col_idx, (title, plot_func) in enumerate(plot_types):
        ax = axes[row_idx, col_idx]
        plot_func(ax, column)  # Llamamos a la función de ploteo
        if col_idx == 0:  
            ax.set_ylabel(feature)  # Etiquetamos la fila con la variable
        ax.set_title(title)  # Ponemos el tipo de gráfico como título

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

Acortar pacientes pedaitricos:

In [ ]:
# Filtrar pacientes menores de 18
menores_18 = patient_data[patient_data["EDAD"] < 18]

# Agrupar y contar cuántas muestras hay por cada edad
desglose_edades = menores_18["EDAD"].value_counts().sort_index()

print("Desglose de muestras por edad (menores de 18):")
print(desglose_edades)

In [ ]:
# Eliminar pacientes menores de 18 años
patient_data = patient_data[patient_data["EDAD"] >= 18].reset_index(drop=True)
N_pat, N_columns = patient_data.shape
print(f"Number of patients: {N_pat}")

In [ ]:
patient_data.loc[patient_data['numero'] == 0, 'numero'] = np.nan
patient_data['numero'].unique()

In [ ]:
features = ['BMI','diametro_mayor','diametro_menor','numero','UH',
            'distanciapielcalculo','EDAD'] + ['calibre_vaina','duracion_tto']

fig, axes = plt.subplots(nrows=len(features), ncols=2, figsize=(10, 3 * len(features)))
fig.suptitle("Distribuciones de caracteristicas numericas", fontsize=16)

plot_types = [
    ("Boxplot", lambda ax, data: ax.boxplot(data)),
    ("Histogram", lambda ax, data: ax.hist(data, bins=20, edgecolor='black'))
]

for row_idx, feature in enumerate(features):
    column = patient_data[feature].dropna().values
    for col_idx, (title, plot_func) in enumerate(plot_types):
        ax = axes[row_idx, col_idx]
        plot_func(ax, column)
        if col_idx == 0:
            ax.set_ylabel(feature)
        ax.set_title(title)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
#!pip install seaborn

# Arreglo de <N/A>

In [ ]:
patient_data = patient_data.replace({pd.NA: np.nan})

In [ ]:
print(patient_data[patient_data["codigo"] == 'NLP444'])

In [ ]:
type_value = type(patient_data.loc[patient_data["codigo"] == 'NLP444', 'Cultresul'].iloc[0])
print(type_value)

In [ ]:
import pandas as pd
import numpy as np

patient_data['Cultresul'] = patient_data['Cultresul'].apply(
    lambda x: np.nan if pd.isna(x) else x
)

In [ ]:
print(patient_data[patient_data["codigo"] == 'NLP444'])

In [ ]:
# todo el DataFrame
patient_data = patient_data.astype('object')
patient_data = patient_data.applymap(lambda x: np.nan if pd.isna(x) else x)

In [ ]:
patient_data.tail()

# Crear la columna de label Complicación de sangrados

In [ ]:
patient_data.isna().sum()

In [ ]:
# 'HEMATURIA','HEMATOMA','SANGRADO','trasfusion','embolizacion'
patient_data.dropna(subset=['HEMATURIA'],inplace=True)
patient_data.dropna(subset=['HEMATOMA'],inplace=True)
patient_data.dropna(subset=['SANGRADO'],inplace=True)
patient_data.dropna(subset=['trasfusion'],inplace=True)
patient_data.dropna(subset=['embolizacion'],inplace=True)

In [ ]:
patient_data.tail()

In [ ]:
patient_data['ComplHemo'] = ((patient_data['HEMATURIA'] == 1) | (patient_data['HEMATOMA'] == 1)| (patient_data['SANGRADO'] == 1)| (patient_data['trasfusion'] == 1)| (patient_data['embolizacion'] == 1)).astype(int)

In [ ]:
patient_data.head()

In [ ]:
# Contar cuántas veces aparece el valor 1 en las columnas específicas
columnas = ['HEMATURIA','HEMATOMA','SANGRADO','trasfusion','embolizacion','ComplHemo']
conteos = patient_data[columnas].apply(lambda col: (col == 1).sum())

# Mostrar los resultados
print(conteos)

In [ ]:
patient_data.drop(columns=['HEMATURIA','HEMATOMA','SANGRADO','trasfusion','embolizacion'], inplace=True)

In [ ]:
patient_data.head()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
# Mapeo 1 -> Sí, 0 -> No
serie = patient_data["ComplHemo"].map({1: "Sí", 0: "No"})
# Ignorar NaN explícitamente (comportamiento tipo SPSS: listwise deletion)
serie = serie.dropna()
# Conteo
counts = serie.value_counts()
# Pie chart
plt.figure(figsize=(6,6))
plt.pie(
    counts,
    labels=counts.index,
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Complicaciones de origen hemorrágica")
plt.legend(title="Complicaciones de origen hemorrágica")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

feat_cat = 'ComplHemo'

# Lista para guardar resultados
results = []

n_cols = 3
n_rows = -(-len(numericas) // n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 4))
axes = axes.flatten()

for idx, feature in enumerate(numericas):

    subset = patient_data[[feat_cat, feature]].dropna()

    num_cat0 = subset.loc[subset[feat_cat] == 0, feature]
    num_cat1 = subset.loc[subset[feat_cat] == 1, feature]

    # Saltar variables sin datos suficientes
    if num_cat0.empty or num_cat1.empty:
        axes[idx].set_visible(False)
        continue

    # Mann-Whitney U
    stat, p_value = stats.mannwhitneyu(
        num_cat0,
        num_cat1,
        alternative='two-sided'
    )

    # Significancia
    sig = (
        '***' if p_value < 0.001 else
        '**' if p_value < 0.01 else
        '*' if p_value < 0.05 else
        ''
    )

    # Guardar resultados
    results.append({
        'Variable': feature,
        'p_value': round(p_value, 4),
        'Significativo': sig
    })

    # Boxplot
    axes[idx].boxplot(
        [num_cat0, num_cat1],
        labels=['No', 'Sí']
    )

    axes[idx].set_title(f"Distribución de {feature}")
    axes[idx].set_xlabel(f"Mann-Whitney p={p_value:.4f} {sig}")

# Ocultar ejes vacíos
for j in range(idx + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout(h_pad=2)
plt.show()

# Crear DataFrame de resultados
results_df = pd.DataFrame(results)

# Mostrar resultados
results_df

In [ ]:
categoricas = [
    col for col in patient_data.columns
    if col not in numericas and col not in ['codigo','ComplHemo']
]

In [ ]:
mapeos = {
    'tecnica': {
        0.0: "estándar",
        1.0: "mini-perc",
        2.0: "combinado flexible anterógrado",
        3.0: "combinado flexible retrógrado",
        4.0: "combinado semirrígido retrógrado",
        5.0: "combinado anterógrado retrógrado",
        6.0: "micro-perc"
    },

    # =========================
    # PREOPERATORIAS
    # =========================

    'Sexo': {
        0: 'Hombre',
        1: 'Mujer'
    },

    'Diabetes': {
        0: 'No',
        1: 'Insulinodependiente',
        2: 'No insulinodependiente'
    },

    'HTA': {
        0: 'No',
        1: 'Sí'
    },

    'Fumador_tabaco': {
        0: 'No',
        1: 'Sí',
        2: 'Exfumador'
    },

    'ASA': {
        1: 'ASA I',
        2: 'ASA II',
        3: 'ASA III',
        4: 'ASA IV'
    },

    'Cultresul': {
        0: 'Negativo',
        1: 'Positivo',
        2: 'Contaminado'
    },

    'Microorganismo': {
        0: 'Negativo',
        1: 'Ureasa negativo',
        2: 'Ureasa positivo'
    },

    'tipotto': {
        0: 'Primario',
        1: 'Post-ESWL',
        2: 'Post-URS',
        3: 'Post-NLP'
    },

    'lado': {
        0: 'Derecho',
        1: 'Izquierdo',
        2: 'Bilateral',
        3: 'Trasplante'
    },
    'localizacion': {
        0: 'Pelvis',
        1: 'Caliz superior',
        2: 'Caliz medio',
        3: 'Caliz inferior',
        4: 'Pseudocoraliforme',
        5: 'Coraliforme',
        6: 'Renal + ureteral',
        7: 'Diverticulo calicial',
        8: 'Pelvis + GCI',
        9: 'Multiples calices',
        10: 'Ureter proximal',
        11: 'Ureter distal',
        12: 'Pelvis + GCS'
    },

    'Guy_Score': {
        1: 'I',
        2: 'II',
        3: 'III',
        4: 'IV'
    },
    # =========================
    # INTRAOPERATORIAS
    # =========================

    'posicion_paciente': {
        0: 'Prono',
        1: 'Supino'
    },

    'acceso_puncion': {
        0: 'Bajo escopia',
        1: 'Ecografía',
        2: 'Ambas'
    },

    'cateterismo_ureteral': {
        0: 'No',
        1: 'Sí'
    },

    'contraste': {
        0: 'No',
        1: 'Sí'
    },

    'metodo_dilatacion': {
        0: 'Amplatz',
        1: 'Balón',
        2: 'Ambos',
        3: 'Metálicos'
    },

    'multitrayecto': {
        0: 'No',
        1: 'Sí'
    },

    'calibre_vaina': {
        0: '9-11',
        1: '10-12'
    },

    'fuente_fragmentacion': {
        0: 'Lithoclast',
        1: 'Ultrasonido',
        2: 'Lithoclast + Ultrasonido',
        3: 'Láser holmium',
        4: 'Cesta',
        5: 'Pinzas',
        6: 'Lithoclast + Láser',
        7: 'Irrigación'
    },

    'facilidad_localizacion': {
        0: 'Fácil',
        1: 'Media',
        2: 'Difícil'
    },

    'anestesia': {
        0: 'Sedación',
        1: 'General',
        2: 'Local'
    },

    'drenajes': {
        0: 'No',
        1: 'Nefrostomía',
        2: 'Doble J',
        3: 'Doble J + nefrostomía',
        4: 'Catéter ureteral 24h',
        5: 'Catéter ureteral 24h + nefrostomía'
    },

    'tubeless': {
        0: 'No',
        1: 'Sí'
    },
    'Procalcitonina':{
        0.0: 'Rango normal',
        1.0: 'Fuera del rango normal'
    },
    'LeucocitosTotales':{
        0.0: 'Rango normal',
        1.0: 'Fuera del rango normal'
    },
    'ProcentajeNeutrofilos':{
        0.0: 'Rango normal',
        1.0: 'Fuera del rango normal'
    }
}

In [ ]:
patient_data.columns

In [ ]:
categoricas

In [ ]:
from scipy.stats import chi2_contingency
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import math

feat_cat1 = 'ComplHemo'
results = []

# =========================================================
# FUNCIÓN FORMATO P-VALOR
# =========================================================
def format_pvalue(p):
    if p < 0.0001:
        return "<0.0001"
    return f"{p:.4f}"

# Variables que irán en figura aparte conjunta
vars_separadas = ['localizacion', 'tecnica', 'fuente_fragmentacion', 'drenajes', 'metodo_dilatacion']

# Variables para figura principal
categoricas_orden = [c for c in categoricas if c not in vars_separadas]

# =========================================================
# FILTRADO VARIABLES FIGURA PRINCIPAL
# =========================================================
valid_vars_main = []

for feat_cat2 in categoricas_orden:
    subset = patient_data[[feat_cat1, feat_cat2]].dropna()

    if subset.empty or subset[feat_cat2].nunique() < 2:
        continue

    valid_vars_main.append(feat_cat2)

# =========================================================
# FIGURA PRINCIPAL (2 COLUMNAS)
# =========================================================
n_cols = 2
n_rows = math.ceil(len(valid_vars_main) / n_cols)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(n_cols * 6, n_rows * 4)
)

axes = axes.flatten()

for i, feat_cat2 in enumerate(valid_vars_main):

    subset = patient_data[[feat_cat1, feat_cat2]].dropna().copy()

    subset[feat_cat1] = subset[feat_cat1].map(
        mapeos.get(feat_cat1, lambda x: str(x))
    )

    subset[feat_cat2] = subset[feat_cat2].map(
        mapeos.get(feat_cat2, lambda x: str(x))
    )

    raw_crosstab = pd.crosstab(
        subset[feat_cat1],
        subset[feat_cat2]
    )

    norm_crosstab = pd.crosstab(
        subset[feat_cat1],
        subset[feat_cat2],
        normalize='index'
    ) * 100

    _, pvalue, _, _ = chi2_contingency(raw_crosstab)

    p_text = format_pvalue(pvalue)

    sns.heatmap(
        norm_crosstab,
        ax=axes[i],
        annot=raw_crosstab,
        fmt='d',
        cmap='Blues',
        linewidths=0.5,
        cbar=False,
        vmin=0,
        vmax=100
    )

    axes[i].set_title(f"{feat_cat2}  p={p_text}")
    axes[i].set_xlabel('')
    axes[i].set_ylabel(feat_cat1)

    results.append({
        'Variable': feat_cat2,
        'p-valor': p_text
    })

# Ocultar ejes sobrantes
for j in range(len(valid_vars_main), len(axes)):
    axes[j].set_visible(False)

plt.suptitle(
    f'Tabla de contingencia vs {feat_cat1}',
    fontsize=14,
    y=0.98
)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# =========================================================
# FIGURA APARTE PARA LAS 4 VARIABLES ESPECIALES
# =========================================================
valid_vars_sep = []

for var in vars_separadas:

    if var not in patient_data.columns:
        continue

    subset = patient_data[[feat_cat1, var]].dropna()

    if subset.empty or subset[var].nunique() < 2:
        continue

    valid_vars_sep.append(var)

# Crear figura conjunta
n_cols_sep = 2
n_rows_sep = math.ceil(len(valid_vars_sep) / n_cols_sep)

fig, axes = plt.subplots(
    n_rows_sep,
    n_cols_sep,
    figsize=(n_cols_sep * 6, n_rows_sep * 4)
)

axes = axes.flatten()

for i, var in enumerate(valid_vars_sep):

    subset = patient_data[[feat_cat1, var]].dropna().copy()

    subset[feat_cat1] = subset[feat_cat1].map(
        mapeos.get(feat_cat1, lambda x: str(x))
    )

    subset[var] = subset[var].map(
        mapeos.get(var, lambda x: str(x))
    )

    raw_crosstab = pd.crosstab(
        subset[feat_cat1],
        subset[var]
    )

    norm_crosstab = pd.crosstab(
        subset[feat_cat1],
        subset[var],
        normalize='index'
    ) * 100

    _, pvalue, _, _ = chi2_contingency(raw_crosstab)

    p_text = format_pvalue(pvalue)

    sns.heatmap(
        norm_crosstab,
        ax=axes[i],
        annot=raw_crosstab,
        fmt='d',
        cmap='Blues',
        linewidths=0.5,
        cbar=False,
        vmin=0,
        vmax=100
    )

    axes[i].set_title(f"{var}  p={p_text}")
    axes[i].set_xlabel('')
    axes[i].set_ylabel(feat_cat1)

    results.append({
        'Variable': var,
        'p-valor': p_text
    })

# Ocultar sobrantes
for j in range(len(valid_vars_sep), len(axes)):
    axes[j].set_visible(False)

plt.suptitle(
    f'Tabla de contingencia vs {feat_cat1}',
    fontsize=14,
    y=0.98
)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# =========================================================
# RESULTADOS
# =========================================================
results_df = pd.DataFrame(results)

results_df

# Matriz de correlacion

In [ ]:
import seaborn as sns
# Calcular la matriz de correlación
sacar_correlacion = patient_data.drop(columns=['codigo'])
correlacion = sacar_correlacion.corr()
# Visualizar la matriz de correlación con un heatmap
plt.figure(figsize=(20, 10))
sns.heatmap(correlacion, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matriz de Correlación')
plt.show()

Variable anestesia no tiene varianza y todos los valores son "1" (anestesia general), por lo cual se elimina del estudio

In [ ]:
# Verificar si la columna tiene solo NaN
patient_data['anestesia'].isna().sum()

# Verificar si todos los valores son iguales
patient_data['anestesia'].nunique()
patient_data = patient_data.drop(columns=['anestesia'])

# Tratamiento de NaNs

**Eliminación de muestras y caracteristicas con una gran ausencia de datos:**

Cantidad de NaNs por columna:

In [ ]:
patient_data.isna().sum()

In [ ]:
data_nan = np.isnan(patient_data.drop(columns=['codigo']))
nans_per_colum = np.sum(data_nan, axis=0)
plt.xticks(rotation=90)
plt.bar(patient_data.drop(columns=['codigo']).columns, nans_per_colum)
plt.show()

Cantidad de NaNs por fila (elemento):

In [ ]:
# Contar los NaNs por fila
nans_por_fila = patient_data.isna().sum(axis=1)
# Filtrar solo las filas que tienen más de 0 NaNs
nans_por_fila_con_na = nans_por_fila[nans_por_fila > 0]
#print(nans_por_fila_con_na)
print('Total de elementos con al menos un NaN:',len(nans_por_fila_con_na)) 
cantidad_por_nans = nans_por_fila.value_counts().sort_index()
print('La cantidad de Nans en una fila y el número de filas con esta cantidad:')
print(cantidad_por_nans)

"Una regla común es eliminar pacientes que tengan más de un 30-40% de datos faltantes."

In [ ]:
len(patient_data.columns)

In [ ]:
#umbral_eliminar_filas_por_Nan = 0.3*len(patient_data.columns) #quizas cuando haya mas car-cas y la distribucion de NaNs sea diferente cambiar el umbral 0,4*
umbral_eliminar_filas_por_Nan =9

Se eliminan los pacientes que tengan un número de NaNs mayor que umbral (30%):

In [ ]:
nans_per_row = np.sum(data_nan, axis=1)
index_rows = np.argwhere(nans_per_row.to_numpy() > umbral_eliminar_filas_por_Nan)[:,0]
patient_data_clean = patient_data.drop(patient_data.index[index_rows])
N_pat, N_columns = patient_data_clean.shape
print(f"Number of patients: {N_pat}")
print(f"Number of columns: {N_columns}")

**Vuelvo a quitar decimal sobrante de variables categoricas**

In [ ]:
print(patient_data_clean.columns.tolist())

In [ ]:
# quitar decimal sobrante de variables categoricas
# ###categorias = ['Sexo', 'Diabetes', 'ASA', 'tipotto', 'numero', 'Guy_Score',  'cateterismo_ureteral', 'contraste', 'multitrayecto', 'facilidad_localizacion', 'tubeless', 'Procalcitonina', 'LeucocitosTotales', 'ProcentajeNeutrofilos', 'IngresoSiNo', 'BMI_missing', 'Cultresul_missing', 'Microorganismo_missing', 'diametro_mayor_missing', 'diametro_menor_missing', 'numero_missing', 'localizacion_missing', 'Guy_Score_missing', 'UH_missing', 'distanciapielcalculo_missing', 'tecnica_missing', 'posicion_paciente_missing', 'acceso_puncion_missing', 'contraste_missing', 'metodo_dilatacion_missing', 'multitrayecto_missing', 'calibre_vaina_missing', 'fuente_fragmentacion_missing', 'facilidad_localizacion_missing', 'duracion_tto_missing', 'Cultresul_0.0', 'Cultresul_1.0', 'Cultresul_2.0', 'Microorganismo_0.0', 'Microorganismo_1.0', 'Microorganismo_2.0', 'lado_0', 'lado_1', 'lado_2', 'lado_3', 'localizacion_0.0', 'localizacion_1.0', 'localizacion_2.0', 'localizacion_3.0', 'localizacion_4.0', 'localizacion_5.0', 'localizacion_6.0', 'localizacion_7.0', 'localizacion_8.0', 'localizacion_9.0', 'localizacion_10.0', 'localizacion_11.0', 'localizacion_12.0', 'tecnica_0.0', 'tecnica_1.0', 'tecnica_2.0', 'tecnica_3.0', 'tecnica_4.0', 'tecnica_5.0', 'tecnica_6.0', 'posicion_paciente_0.0', 'posicion_paciente_1.0', 'acceso_puncion_0.0', 'acceso_puncion_1.0', 'acceso_puncion_2.0', 'metodo_dilatacion_0.0', 'metodo_dilatacion_1.0', 'metodo_dilatacion_2.0', 'metodo_dilatacion_3.0', 'fuente_fragmentacion_0.0', 'fuente_fragmentacion_1.0', 'fuente_fragmentacion_2.0', 'fuente_fragmentacion_3.0', 'fuente_fragmentacion_4.0', 'fuente_fragmentacion_5.0', 'fuente_fragmentacion_6.0', 'fuente_fragmentacion_7.0', 'drenajes_0.0', 'drenajes_1.0', 'drenajes_2.0', 'drenajes_3.0', 'drenajes_4.0', 'drenajes_5.0']	
# ### 
categorias = ['Sexo', 'HTA', 'Fumador_tabaco','Diabetes', 'ASA', 'Cultresul', 'Microorganismo', 'tipotto', 'lado', 'numero', 'localizacion', 'Guy_Score', 'EDAD', 'tecnica', 'posicion_paciente', 'acceso_puncion', 'cateterismo_ureteral', 'contraste', 'metodo_dilatacion', 'multitrayecto', 'fuente_fragmentacion', 'facilidad_localizacion', 'duracion_tto', 'drenajes', 'tubeless', 'Procalcitonina', 'LeucocitosTotales', 'ProcentajeNeutrofilos']
patient_data_clean[categorias] = patient_data_clean[categorias].astype('Int64')
patient_data_clean.head()

# Tratamiento de variables categoricas

Ordenar bien la variable categorica de Diabetes para poder utilizarla como ordinal y no nominal:

Originalmente:
0 NO
1 DM INSULINO
2 DM NO INSULINO

Nuevo:
0 NO
1 DM NO INSULINO
2 DM INSULINO

In [ ]:
patient_data_clean["Diabetes"] = patient_data_clean["Diabetes"].replace({1: 2, 2: 1})
#Comprobación:
print('2->1')
print('Nuevo valor:', patient_data_clean.loc[patient_data_clean["codigo"] == 'NLP743', "Diabetes"].values)
print('Valor anterior:', patient_data.loc[patient_data["codigo"] == 'NLP743', "Diabetes"].values)
print('1->2')
print('Nuevo valor:', patient_data_clean.loc[patient_data_clean["codigo"] == 'NLP425', "Diabetes"].values)
print('Valor anterior:', patient_data.loc[patient_data["codigo"] == 'NLP425', "Diabetes"].values)

One-hot-encoding: (mirar cuales son nominales y cuales son ordinales)

In [ ]:
# Uso de one-hot-encoding para variables nominales (NO ordinales) y no binarias:
#patient_data_clean = pd.get_dummies(patient_data_clean, columns = ["Diabetes"], dtype= 'int') #ordinal cambiar "DM NO INSULINO": 1, "DM INSULINO": 2?
#patient_data_clean = pd.get_dummies(patient_data_clean, columns = ["ASA"], dtype= 'int') #ordinal
patient_data_clean = pd.get_dummies(patient_data_clean, columns = ["Cultresul"], dtype= 'int')
patient_data_clean = pd.get_dummies(patient_data_clean, columns = ["Microorganismo"], dtype= 'int')
### patient_data_clean = pd.get_dummies(patient_data_clean, columns = ["motivo"], dtype= 'int')
#patient_data_clean = pd.get_dummies(patient_data_clean, columns = ["tipotto"], dtype= 'int') #ordinal
patient_data_clean = pd.get_dummies(patient_data_clean, columns = ["lado"], dtype= 'int')
patient_data_clean = pd.get_dummies(patient_data_clean, columns = ["localizacion"], dtype= 'int')
#patient_data_clean = pd.get_dummies(patient_data_clean, columns = ["Guy_Score"], dtype= 'int') #ordinal

patient_data_clean = pd.get_dummies(patient_data_clean, columns = ["tecnica"], dtype= 'int')
patient_data_clean = pd.get_dummies(patient_data_clean, columns = ["posicion_paciente"], dtype= 'int')
patient_data_clean = pd.get_dummies(patient_data_clean, columns = ["acceso_puncion"], dtype= 'int')
patient_data_clean = pd.get_dummies(patient_data_clean, columns = ["metodo_dilatacion"], dtype= 'int')
patient_data_clean = pd.get_dummies(patient_data_clean, columns = ["fuente_fragmentacion"], dtype= 'int')
#patient_data_clean = pd.get_dummies(patient_data_clean, columns = ["anestesia"], dtype= 'int') <-realmente todos utilizan la general en la BBDD
patient_data_clean = pd.get_dummies(patient_data_clean, columns = ["drenajes"], dtype= 'int')

patient_data_clean = pd.get_dummies(patient_data_clean, columns = ["Fumador_tabaco"], dtype= 'int')

In [ ]:
patient_data_clean.head()

In [ ]:
print(patient_data_clean.columns)

**Arreglar acceso_puncion**

In [ ]:
# Si acceso_puncion_2 == 1 (ambas tecnicas), asignar 1 a las otras dos columnas
patient_data_clean.loc[patient_data_clean['acceso_puncion_2'] == 1, ['acceso_puncion_0', 'acceso_puncion_1']] = 1
patient_data_clean.drop(columns='acceso_puncion_2', inplace=True)

**Arreglar drenajes**

In [ ]:
patient_data_clean.loc[patient_data_clean['drenajes_3'] == 1, ['drenajes_1', 'drenajes_2']] = 1
patient_data_clean.loc[patient_data_clean['drenajes_5'] == 1, ['drenajes_1', 'drenajes_4']] = 1
patient_data_clean.drop(columns='drenajes_3', inplace=True)
patient_data_clean.drop(columns='drenajes_5', inplace=True)

**Arreglar fuente_fragmentacion**

In [ ]:
patient_data_clean.loc[patient_data_clean['fuente_fragmentacion_2'] == 1, ['fuente_fragmentacion_0', 'fuente_fragmentacion_1']] = 1
patient_data_clean.loc[patient_data_clean['fuente_fragmentacion_6'] == 1, ['fuente_fragmentacion_0', 'fuente_fragmentacion_3']] = 1
patient_data_clean.drop(columns='fuente_fragmentacion_2', inplace=True)
patient_data_clean.drop(columns='fuente_fragmentacion_6', inplace=True)


**Agrupamiento más general para columna localizacion:**

In [ ]:
cols_pelvis = ['localizacion_0', 'localizacion_4', 'localizacion_5', 'localizacion_8', 'localizacion_12']
### Crear la nueva columna loc_pelvis
patient_data_clean['loc_pelvis'] = (patient_data_clean[cols_pelvis] == 1).any(axis=1).astype(int)
patient_data_clean['loc_caliz_sup'] = (patient_data_clean[['localizacion_1']] == 1).any(axis=1).astype(int)
patient_data_clean['loc_caliz_med'] = (patient_data_clean[['localizacion_2']] == 1).any(axis=1).astype(int)
patient_data_clean['loc_caliz_inf'] = (patient_data_clean[['localizacion_3']] == 1).any(axis=1).astype(int)
patient_data_clean['loc_renal y ureteral'] = (patient_data_clean[['localizacion_6']] == 1).any(axis=1).astype(int)
patient_data_clean['ureteral'] = (patient_data_clean[['localizacion_10','localizacion_11']] == 1).any(axis=1).astype(int)

**'localizacion_9'**

In [ ]:
cols_to_update = ['loc_caliz_inf', 'loc_caliz_sup', 'loc_caliz_med']
patient_data_clean.loc[patient_data_clean['localizacion_9'] == 1, cols_to_update] = 1

Localizacion_7

In [ ]:
patient_data_clean.loc[patient_data_clean['codigo'] == "NLP097", 'loc_caliz_inf'] = 1
patient_data_clean.loc[patient_data_clean['codigo'] == "NLP113", 'loc_caliz_inf'] = 1
patient_data_clean.loc[patient_data_clean['codigo'] == "NLP441", 'loc_caliz_inf'] = 1
patient_data_clean.loc[patient_data_clean['codigo'] == "NLP117", 'loc_caliz_sup'] = 1
patient_data_clean.loc[patient_data_clean['codigo'] == "NLP119", 'loc_caliz_sup'] = 1
patient_data_clean.loc[patient_data_clean['codigo'] == "NLP300", 'loc_caliz_sup'] = 1
patient_data_clean.loc[patient_data_clean['codigo'] == "NLP328", 'loc_caliz_sup'] = 1
patient_data_clean.loc[patient_data_clean['codigo'] == "NLP157", 'loc_caliz_med'] = 1

In [ ]:
patient_data_clean.head()

In [ ]:
###eliminar columnas originales de localización (excepto loc_7 y loc_9 de momemnto):
cols_to_drop = [
    'localizacion_0', 'localizacion_1', 'localizacion_2', 'localizacion_3',
    'localizacion_4', 'localizacion_5', 'localizacion_6', 'localizacion_7',
    'localizacion_8', 'localizacion_9', 'localizacion_10', 'localizacion_11', 'localizacion_12'
]
patient_data_clean = patient_data_clean.drop(columns=cols_to_drop)

In [ ]:
print(patient_data_clean.columns)

In [ ]:
patient_data_clean.head()

In [ ]:
#patient_data_clean.to_excel("23_patient_data_clean.xlsx", index=False)

# Data splitting

Tratamiento de pacientes repetidos
(from Lab1 script:"In cases with multiple measures for each patient (each patient is present in multiple 
rows), each patient must belong to a single set. Measures of the same patient cannot 
be present on multiple sets (training, validation or test). After the data splitting, remove 
the patient-ID column.")

In [ ]:
# Contar cuántas veces aparece cada ID de paciente en la columna 'n_historia'
repeated_patients = patient_data_clean['codigo'].value_counts()
# Filtrar solo aquellos que se repiten más de una vez (mayor a 1)
repeated_patients = repeated_patients[repeated_patients > 1]
# Contar la frecuencia de cada número de repeticiones
repetition_counts = repeated_patients.value_counts().sort_index()
# Mostrar cuántos pacientes hay con 2 repeticiones, 3 repeticiones, etc.
print(repetition_counts)
# Mostrar la suma total de pacientes repetidos
print("Total de pacientes repetidos:", repetition_counts.sum())

Separación de datos en train, validation y test 70%/15%/15%. Los mismos pacientes siempre están en el mismo set.
También se utiliza StratifiedGroupKFold para garantizar que la proporción entre clases IngresoSi y IngresoNo sea igual en los 3 subconjuntos, es decir, la **ESTRATIFICACIÓN**.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
# split 80% y 20%
# Variables 
X = patient_data_clean.drop(columns=['ComplHemo', 'codigo'])
y = patient_data_clean['ComplHemo']
groups = patient_data_clean['codigo'].values

# 5 folds -> 1 fold test = 20%
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=2 #2 #5
)

# Primer split
train_idx, test_idx = next(sgkf.split(X, y, groups))

# Train/Test
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups[train_idx]
groups_test = groups[test_idx]

print("Train size:", len(X_train))
print("Test size:", len(X_test))

(X_train, X_test, y_train, y_test, groups_train, groups_test)

In [ ]:
#Visualización del resultado y comprobación de ausencia de n_historia en los sets:
X_test.head()

In [ ]:
# Crear DataFrame de entrenamiento, validación y test
train_data = pd.concat([X_train, y_train], axis=1)
# val_data = pd.concat([X_val, y_val], axis=1)
test_data = pd.concat([X_test, y_test], axis=1)
train_data.head()

# EDA

**Variables categoricas sin one-hot-encoding**

In [ ]:
numericas = ['BMI','diametro_mayor','diametro_menor','UH','distanciapielcalculo']+['numero','EDAD']+['calibre_vaina','duracion_tto']

In [ ]:
# Función para calcular estadísticas descriptivas

def calcular_estadisticas(df, cat_cols):
    stats = {}
    # Para variables categóricas: proporciones
    for col in cat_cols:
        proportions = round(df[col].value_counts(normalize=True) * 100,2)
        for value, proportion in proportions.items():
            stats[f"{col}_{value}"] = {'proportion': proportion}

    return pd.DataFrame(stats).T

# Definir columnas categóricas
one_hot_cols = ['Cultresul','Microorganismo', 'lado','localizacion','tecnica','posicion_paciente','acceso_puncion','metodo_dilatacion','fuente_fragmentacion','drenajes']
#cat_cols = ['Sexo', 'Diabetes', 'ASA', 'tipotto','Guy_Score','IngresoSiNo']
# Unir ambas listas de columnas conocidas
columnas_conocidas = set(one_hot_cols + numericas)
columnas_conocidas.add('codigo')
# Filtrar columnas del DataFrame que no están en las listas
cat_cols = [col for col in patient_data_clean.columns if col not in columnas_conocidas]

# Calcular estadísticas para cada conjunto
stats_train = calcular_estadisticas(train_data, cat_cols)
# stats_valid = calcular_estadisticas(val_data, cat_cols)
stats_test = calcular_estadisticas(test_data, cat_cols)

# Combinar las estadísticas en una tabla
stats_summary = pd.concat([stats_train, stats_test], axis=1, keys=['Train', 'Test'])
# stats_summary = pd.concat([stats_train, stats_valid, stats_test], axis=1, keys=['Train', 'Validation', 'Test'])

print(stats_summary)

In [ ]:
print(stats_summary[-91:])

**Variables numericas**

In [ ]:
# Función para calcular media, desviación estándar y mediana (IQR) con redondeo
def calculate_statistics(df):
    stats = {}
    for column in numericas:
        mean = round(df[column].mean(), 2)
        std = round(df[column].std(), 2)
        median = round(df[column].median(), 2)
        q1 = df[column].quantile(0.25)
        q3 = df[column].quantile(0.75)
        iqr = round(q3 - q1, 2)

        stats[column] = {
            'mean': mean,
            'std': std,
            'median': median,
            'IQR': iqr
        }
    return pd.DataFrame(stats)

numericas = ['BMI','diametro_mayor','diametro_menor','UH','distanciapielcalculo']+['numero','EDAD']+['calibre_vaina','duracion_tto']

# Calcular estadísticas para los tres conjuntos de datos
train_stats = calculate_statistics(train_data)
# validation_stats = calculate_statistics(val_data)
test_stats = calculate_statistics(test_data)
pd.concat([train_stats,test_stats], axis=1, keys=['Train', 'Test'])

**Variables con one-hot encoding**

In [ ]:
# Función para calcular las proporciones de variables después de One-Hot Encoding
def calcular_proporciones_one_hot(df, one_hot_cols):
    proportions = {}
    for col in one_hot_cols:
        # Obtenemos las columnas correspondientes a esta variable después de One-Hot Encoding
        one_hot_columns = [c for c in df.columns if c.startswith(col)]
        
        # Calculamos las proporciones para cada una de las columnas de One-Hot Encoding
        for o_col in one_hot_columns:
            count_ones = df[o_col].sum()  # Contamos cuántos 1s hay en esa columna
            total = df[o_col].shape[0]  # Total de registros en esa columna
            proportion = round((count_ones / total) * 100 ,2 ) # Proporción en porcentaje
            proportions[f"{o_col}_proportion"] = proportion
    
    # Convertimos el diccionario en un DataFrame
    proportions_df = pd.DataFrame(proportions, index=[0])
    return proportions_df

# Ejemplo de uso: Definir columnas One-Hot Encoding
one_hot_cols = ['Cultresul','Microorganismo', 'lado','localizacion','tecnica','posicion_paciente','acceso_puncion','metodo_dilatacion','fuente_fragmentacion','drenajes']
# Calcular estadísticas para cada conjunto
stats_train = calcular_proporciones_one_hot(train_data, one_hot_cols)
# stats_valid = calcular_proporciones_one_hot(val_data, one_hot_cols)
stats_test = calcular_proporciones_one_hot(test_data, one_hot_cols)
df_1hot_eda=pd.concat([stats_train, stats_test], axis=0, keys=['Train', 'Test'])
# df_1hot_eda=pd.concat([stats_train, stats_valid, stats_test], axis=0, keys=['Train', 'Validation', 'Test'])

pd.DataFrame(df_1hot_eda).T

In [ ]:
#!pip install imbalanced-learn

# Imputar valores faltantes

In [ ]:
X_test.isna().sum()

In [ ]:
# Columnas numéricas
numeric_cols = ['EDAD','BMI','diametro_mayor','diametro_menor','UH',
                'distanciapielcalculo','duracion_tto']
                # ,'Procalcitonina','LeucocitosTotales','ProcentajeNeutrofilos']

# Columnas categóricas
categorical_cols = [col for col in X_train.columns if col not in numeric_cols and col != "IngresoSiNo"]

# ===== NUMÉRICAS: mediana calculada SOLO en X_train =====
median_values = X_train[numeric_cols].median()

X_train[numeric_cols] = X_train[numeric_cols].fillna(np.round(median_values))
X_test[numeric_cols]  = X_test[numeric_cols].fillna(np.round(median_values))

# ===== CATEGÓRICAS: moda calculada SOLO en X_train =====
mode_values = X_train[categorical_cols].mode().iloc[0]

X_train[categorical_cols] = X_train[categorical_cols].fillna(mode_values)
X_test[categorical_cols]  = X_test[categorical_cols].fillna(mode_values)

In [ ]:
median_values

In [ ]:
mode_values

In [ ]:
X_test.isna().sum()[:-30]

# Entrenar Modelos ML

#### fucnion

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    make_scorer,
    cohen_kappa_score,
    matthews_corrcoef
)

# =========================================================
# FUNCIÓN GENERAL
# =========================================================

def random_search_cv_model(
    model,
    param_dist,
    X_train,
    y_train,
    groups_train,
    refit_metric="auc",
    n_iter=30,
    scale_data=False,
    random_state=42,
    n_jobs=-1,
    verbose=1
):
    """
    Ejecuta RandomizedSearchCV con múltiples métricas.

    Parámetros
    ----------
    model : estimator
        Modelo sklearn (DecisionTree, RandomForest, LogisticRegression, XGBoost, etc.)

    param_dist : dict
        Espacio de hiperparámetros.

    X_train : array-like
    y_train : array-like
    groups_train : array-like

    refit_metric : str
        Métrica usada para seleccionar el mejor modelo.
        Opciones:
        - "auc"
        - "accuracy"
        - "kappa"
        - "mcc"

    scale_data : bool
        Si True, aplica MinMaxScaler mediante Pipeline.
        Útil para Logistic Regression, SVM, etc.

    Returns
    -------
    best_model : estimator
        Mejor modelo ajustado.

    search : RandomizedSearchCV
        Objeto completo del search.
    """

    # =====================================================
    # SCORERS
    # =====================================================

    scoring = {
        "auc": "roc_auc",
        "accuracy": "accuracy",
        "kappa": make_scorer(cohen_kappa_score),
        "mcc": make_scorer(matthews_corrcoef)
    }

    # =====================================================
    # PIPELINE OPCIONAL
    # =====================================================

    if scale_data:

        estimator = Pipeline([
            ("scaler", MinMaxScaler()),
            ("model", model)
        ])

    else:

        estimator = model

    # =====================================================
    # CROSS VALIDATION
    # =====================================================

    cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=random_state
    )

    # =====================================================
    # RANDOM SEARCH
    # =====================================================

    search = RandomizedSearchCV(
        estimator=estimator,

        param_distributions=param_dist,

        n_iter=n_iter,

        scoring=scoring,

        refit=refit_metric,

        cv=cv,

        n_jobs=n_jobs,

        verbose=verbose,

        random_state=random_state
    )

    # =====================================================
    # ENTRENAMIENTO
    # =====================================================

    search.fit(
        X_train,
        y_train,
        groups=groups_train
    )

    # =====================================================
    # RESULTADOS
    # =====================================================

    best_idx = search.best_index_

    # ===== AUC =====

    mean_auc = search.cv_results_["mean_test_auc"][best_idx]
    std_auc = search.cv_results_["std_test_auc"][best_idx]

    # ===== ACCURACY =====

    mean_acc = search.cv_results_["mean_test_accuracy"][best_idx]
    std_acc = search.cv_results_["std_test_accuracy"][best_idx]

    # ===== KAPPA =====

    mean_kappa = search.cv_results_["mean_test_kappa"][best_idx]
    std_kappa = search.cv_results_["std_test_kappa"][best_idx]

    # ===== MCC =====

    mean_mcc = search.cv_results_["mean_test_mcc"][best_idx]
    std_mcc = search.cv_results_["std_test_mcc"][best_idx]

    print("\n=========================")
    print("RESULTADOS CV")
    print("=========================\n")

    print(f"AUC CV = {mean_auc:.3f} ± {std_auc:.3f}")
    print(f"Accuracy CV = {mean_acc:.3f} ± {std_acc:.3f}")
    print(f"Kappa CV = {mean_kappa:.3f} ± {std_kappa:.3f}")
    print(f"MCC CV = {mean_mcc:.3f} ± {std_mcc:.3f}")

    print("\n=========================")
    print("MEJORES HIPERPARÁMETROS")
    print("=========================\n")

    print(search.best_params_)

    # =====================================================
    # MEJOR MODELO
    # =====================================================

    best_model = search.best_estimator_

    summary = {
        "model": type(model).__name__,

        "refit_metric": refit_metric,

        "auc_mean": mean_auc,
        "auc_std": std_auc,

        "accuracy_mean": mean_acc,
        "accuracy_std": std_acc,

        "kappa_mean": mean_kappa,
        "kappa_std": std_kappa,

        "mcc_mean": mean_mcc,
        "mcc_std": std_mcc,

        "best_params": search.best_params_
    }

    return best_model, search, summary

#### resultados

In [ ]:
results = []

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier()

param_dist = {
    "max_depth": [2, 3, 5, 10, None],
    "min_samples_leaf": [1, 5, 10, 20, 50],
    "min_samples_split": [2, 5, 10, 20],
    "class_weight": [{0: 1, 1: 5}]
}

best_dt_model_auc, search_dt_auc, summary_dt_auc = random_search_cv_model(
    model=tree_model,
    param_dist=param_dist,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    refit_metric="auc",
    scale_data=False
)

results.append(summary_dt_auc)

In [ ]:
best_dt_model_kappa, search_dt_kappa, summary_dt_kappa = random_search_cv_model(
    model=tree_model,
    param_dist=param_dist,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    refit_metric="kappa",
    scale_data=False
)

results.append(summary_dt_kappa)

In [ ]:
best_dt_model_mcc, search_dt_mcc, summary_dt_mcc = random_search_cv_model(
    model=tree_model,
    param_dist=param_dist,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    refit_metric="mcc",
    scale_data=False
)

results.append(summary_dt_mcc)

Random forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    random_state=42
)

# =========================
# ESPACIO DE HIPERPARÁMETROS
# =========================

param_dist = {
    "n_estimators": [10, 50, 100, 200, 300],
    "max_depth": [3, 5, 10, 20, None],
    "min_samples_leaf": [1, 5, 10, 15, 20, 50],
    "min_samples_split": [2, 5, 10, 20],
    "max_features": ["sqrt", "log2", None],
    "class_weight": [{0: 1, 1: 5}]
}

best_rf_model_auc, search_rf_auc, summary_rf_auc = random_search_cv_model(
    model=rf_model,
    param_dist=param_dist,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    refit_metric="auc",
    scale_data=False
)

results.append(summary_rf_auc)

In [ ]:
best_rf_model_kappa, search_rf_kappa, summary_rf_kappa = random_search_cv_model(
    model=rf_model,
    param_dist=param_dist,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    refit_metric="kappa",
    scale_data=False
)

results.append(summary_rf_kappa)

In [ ]:
best_rf_model_mcc, search_rf_mcc, summary_rf_mcc = random_search_cv_model(
    model=rf_model,
    param_dist=param_dist,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    refit_metric="mcc",
    scale_data=False
)

results.append(summary_rf_mcc)

XGboost

In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)

param_dist = {
    "n_estimators": [50, 100, 200, 300],
    
    "max_depth": [2, 3, 5, 7, 10],
    
    "learning_rate": [0.001, 0.01, 0.05, 0.1, 0.2],
    
    "min_child_weight": [1, 5, 10, 20, 35],
    
    "subsample": [0.5, 0.7, 0.8, 1.0],
    
    "colsample_bytree": [0.5, 0.7, 0.8, 1.0],
    
    "gamma": [0, 0.1, 0.3, 1],
    
    "reg_alpha": [0, 0.01, 0.1, 1],
    
    "reg_lambda": [0.1, 1, 5, 10],

    "scale_pos_weight":[5]
}

best_xgb_model_auc, search_xgb_auc, summary_xgb_auc = random_search_cv_model(
    model=xgb_model,
    param_dist=param_dist,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    refit_metric="auc",
    scale_data=False
)

results.append(summary_xgb_auc)

In [ ]:
best_xgb_model_kappa, search_xgb_kappa, summary_xgb_kappa = random_search_cv_model(
    model=xgb_model,
    param_dist=param_dist,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    refit_metric="kappa",
    scale_data=False
)

results.append(summary_xgb_kappa)

In [ ]:
best_xgb_model_mcc, search_xgb_mcc, summary_xgb_mcc = random_search_cv_model(
    model=xgb_model,
    param_dist=param_dist,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    refit_metric="mcc",
    scale_data=False
)

results.append(summary_xgb_mcc)

Regresion logistica

In [ ]:
from sklearn.linear_model import LogisticRegression
import numpy as np

# =====================================================
# MODELO
# =====================================================

logreg_model = LogisticRegression(
    random_state=42,
    max_iter=5000
)

# =====================================================
# HIPERPARÁMETROS
# =====================================================

param_dist = {
    "model__C": np.logspace(-4, 4, 20),

    "model__penalty": ["l1", "l2"],

    "model__solver": ["liblinear"],

    "model__class_weight": [{0: 1, 1: 5}]
}

# =====================================================
# RANDOM SEARCH
# =====================================================

best_logreg_model_auc, search_logreg_auc, summary_logreg_auc = random_search_cv_model(
    model=logreg_model,

    param_dist=param_dist,

    X_train=X_train,

    y_train=y_train,

    groups_train=groups_train,

    refit_metric="auc",

    n_iter=30,

    scale_data=True,

    random_state=42,

    n_jobs=-1,

    verbose=1
)

results.append(summary_logreg_auc)

In [ ]:
best_logreg_model_kappa, search_logreg_kappa, summary_logreg_kappa = random_search_cv_model(
    model=logreg_model, param_dist=param_dist, X_train=X_train, y_train=y_train,
    groups_train=groups_train, refit_metric="kappa", n_iter=30, scale_data=True,
    random_state=42, n_jobs=-1, verbose=1)

results.append(summary_logreg_kappa)

In [ ]:
best_logreg_model_mcc, search_logreg_mcc, summary_logreg_mcc = random_search_cv_model(
    model=logreg_model, param_dist=param_dist, X_train=X_train, y_train=y_train,
    groups_train=groups_train, refit_metric="mcc", n_iter=30, scale_data=True,
    random_state=42, n_jobs=-1, verbose=1)

results.append(summary_logreg_mcc)

# Comparar el rendimiento de los modelos entrenados

In [ ]:
results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
import pandas as pd

# =====================================================
# DATAFRAME
# =====================================================

results_df = pd.DataFrame(results)

# =====================================================
# RENOMBRAR MODELOS
# =====================================================

model_mapping = {
    "DecisionTreeClassifier": "DT",
    "RandomForestClassifier": "RF",
    "XGBClassifier": "XGB",
    "LogisticRegression": "RegLog"
}

results_df["model"] = results_df["model"].map(model_mapping)

# =====================================================
# REGLOG PRIMERO
# =====================================================

model_order = ["RegLog", "DT", "RF", "XGB" ]

results_df["model"] = pd.Categorical(
    results_df["model"],
    categories=model_order,
    ordered=True
)

results_df = results_df.sort_values(
    by=["model", "refit_metric"]
)

# =====================================================
# REDONDEAR
# =====================================================

metric_cols = [
    "auc_mean",
    "auc_std",
    "accuracy_mean",
    "accuracy_std",
    "kappa_mean",
    "kappa_std",
    "mcc_mean",
    "mcc_std"
]

results_df[metric_cols] = results_df[metric_cols].round(3)

# =====================================================
# CONFIGURACIÓN DISPLAY
# =====================================================

pd.set_option("display.max_columns", None)

pd.set_option("display.width", 200)

pd.set_option("display.max_colwidth", 100)

pd.set_option("display.expand_frame_repr", False)

# =====================================================
# PRINT
# =====================================================

print(results_df)

# Prueba final sobre el test set con el mejor modelo, poninedo el umbral personalizado

#### funcion 2 para guardar resultados en el dataframe

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score
)

# =====================================================
# FUNCIÓN DE EVALUACIÓN GENERAL
# =====================================================

def evaluate_model(
    model,
    X_test,
    y_test,
    model_name="Modelo",
    threshold=0.5,
    plot=True
):

    # ==========================================
    # PROBABILIDADES Y PREDICCIONES
    # ==========================================

    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)

    # ==========================================
    # MATRIZ DE CONFUSIÓN
    # ==========================================

    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    # ==========================================
    # MÉTRICAS
    # ==========================================

    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    sensibilidad = recall_score(y_test, y_pred)
    especificidad = tn / (tn + fp) if (tn + fp) > 0 else 0

    vpp = precision_score(y_test, y_pred, zero_division=0)
    vpn = tn / (tn + fn) if (tn + fn) > 0 else 0

    f1 = f1_score(y_test, y_pred, zero_division=0)

    # ==========================================
    # DICCIONARIO RESULTADOS
    # ==========================================

    metrics_dict = {
        "modelo": model_name,
        "umbral": threshold,
        "auc": round(auc, 4),
        "accuracy": round(accuracy, 4),
        "sensibilidad": round(sensibilidad, 4),
        "especificidad": round(especificidad, 4),
        "vpp": round(vpp, 4),
        "vpn": round(vpn, 4),
        "f1_score": round(f1, 4),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

    # ==========================================
    # PRINT RESULTADOS
    # ==========================================

    print("\n" + "=" * 50)
    print(f"EVALUACIÓN DEL MODELO: {model_name}")
    print("=" * 50)

    for key, value in metrics_dict.items():
        if key != "modelo":
            print(f"{key}: {value}")

    # ==========================================
    # GRÁFICOS
    # ==========================================

    if plot:

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        # ------------------------------------------------
        # MATRIZ DE CONFUSIÓN
        # ------------------------------------------------

        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=["Negativo", "Positivo"]
        )

        disp.plot(
            cmap="Blues",
            values_format="d",
            ax=axes[0],
            colorbar=False
        )

        # CAMBIAR ETIQUETAS AL CASTELLANO
        axes[0].set_title(f"Matriz de confusión\n{model_name}")

        axes[0].set_xlabel("Etiqueta predicha")
        axes[0].set_ylabel("Etiqueta real")

        axes[0].grid(False)

        # ------------------------------------------------
        # CURVA ROC
        # ------------------------------------------------

        fpr, tpr, _ = roc_curve(y_test, y_proba)

        axes[1].plot(
            fpr,
            tpr,
            linewidth=2,
            label=f"AUC = {auc:.3f}"
        )

        axes[1].plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            linewidth=1
        )

        axes[1].set_title(f"Curva ROC\n{model_name}")

        axes[1].set_xlabel("Tasa de falsos positivos")
        axes[1].set_ylabel("Tasa de verdaderos positivos")

        axes[1].legend(loc="lower right")
        axes[1].grid(True)

        plt.tight_layout()
        plt.show()

    return metrics_dict

#### Resultados

In [ ]:
results_test = []

In [ ]:
from sklearn.model_selection import TunedThresholdClassifierCV
from sklearn.model_selection import StratifiedGroupKFold
from sklearn import set_config

set_config(enable_metadata_routing=True)

threshold_reglog_model= TunedThresholdClassifierCV(
    estimator=best_logreg_model_auc,

    scoring="balanced_accuracy",   

    cv=StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )
)

threshold_reglog_model.fit(
    X_train,
    y_train,
    groups=groups_train
)

threshold_reglog_model.best_threshold_

In [ ]:
results = evaluate_model(
    model=best_logreg_model_auc,
    X_test=X_test,
    y_test=y_test, model_name="Regresión Logística",
    threshold=threshold_reglog_model.best_threshold_
)
results_test.append(results)

In [ ]:
results = evaluate_model(
    model=best_logreg_model_auc,
    X_test=X_test,
    y_test=y_test, model_name="Regresión Logística",
    threshold=0.5
)
results_test.append(results)

In [ ]:
set_config(enable_metadata_routing=True)

threshold_dt_model= TunedThresholdClassifierCV(
    estimator=best_dt_model_auc,

    scoring="balanced_accuracy",   

    cv=StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )
)

threshold_dt_model.fit(
    X_train,
    y_train,
    groups=groups_train
)

threshold_dt_model.best_threshold_

In [ ]:
results = evaluate_model(
    model=best_dt_model_auc,
    X_test=X_test,
    y_test=y_test, model_name="Árbol de decisión",
    threshold=threshold_dt_model.best_threshold_
)
results_test.append(results)

In [ ]:
results = evaluate_model(
    model=best_dt_model_auc,
    X_test=X_test,
    y_test=y_test, model_name="Árbol de decisión",
    threshold=0.5
)
results_test.append(results)

In [ ]:
from sklearn.model_selection import TunedThresholdClassifierCV
from sklearn.model_selection import StratifiedGroupKFold
from sklearn import set_config

set_config(enable_metadata_routing=True)

threshold_rf_model_auc = TunedThresholdClassifierCV(
    estimator=best_rf_model_auc,

    scoring="balanced_accuracy",   

    cv=StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )
)

threshold_rf_model_auc.fit(
    X_train,
    y_train,
    groups=groups_train
)

threshold_rf_model_auc.best_threshold_

In [ ]:
results = evaluate_model(
    model=best_rf_model_auc,
    X_test=X_test,
    y_test=y_test, model_name="Random Forest",
    threshold=threshold_rf_model_auc.best_threshold_
)
results_test.append(results)

In [ ]:
results = evaluate_model(
    model=best_rf_model_auc,
    X_test=X_test,
    y_test=y_test, model_name="Random Forest",
    threshold=0.5
)
results_test.append(results)

In [ ]:
set_config(enable_metadata_routing=True)

threshold_xgb_model= TunedThresholdClassifierCV(
    estimator=best_xgb_model_kappa,

    scoring="balanced_accuracy",  

    cv=StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )
)

threshold_xgb_model.fit(
    X_train,
    y_train,
    groups=groups_train
)

threshold_xgb_model.best_threshold_

In [ ]:
results = evaluate_model(
    model=best_xgb_model_kappa,
    X_test=X_test,
    y_test=y_test, model_name="XGBoost",
    threshold=threshold_xgb_model.best_threshold_
)
results_test.append(results)

In [ ]:
results = evaluate_model(
    model=best_xgb_model_kappa,
    X_test=X_test,
    y_test=y_test, model_name="XGBoost",
    threshold=0.5
)
results_test.append(results)

In [ ]:
# ==========================================
# DATAFRAME RESUMEN
# ==========================================
summary_df = pd.DataFrame(results_test)
# Ordenar por AUC
summary_df = summary_df.sort_values(
    by="auc",
    ascending=False
)
print(summary_df)

In [ ]:
df_formateado_test = summary_df.iloc[:, :-5].applymap(
    lambda x: f"{x:.4f}".replace(".", ",") if isinstance(x, (int, float)) else x
)
df_formateado_test

La tabla de train:

In [ ]:
df = results_df.iloc[:, :-1].copy()

metricas = ["auc", "accuracy", "kappa", "mcc"]

# empezamos desde el dataframe original pero eliminando columnas mean/std
df_final = df.drop(columns=[f"{m}_mean" for m in metricas] +
                             [f"{m}_std" for m in metricas])

for m in metricas:
    df_final[m] = (
        df[f"{m}_mean"].map(lambda x: f"{x:.3f}".replace(".", ",")) +
        " ± " +
        df[f"{m}_std"].map(lambda x: f"{x:.3f}".replace(".", ","))
    )
df_final

# Shapley

In [ ]:
import re
import shap
# Variables que fueron One-Hot Encoded
ohe_vars = [
    'Cultresul',
    'Microorganismo',
    'lado',
    'localizacion',
    'tecnica',
    'posicion_paciente',
    'acceso_puncion',
    'metodo_dilatacion',
    'fuente_fragmentacion',
    'drenajes',
    'Fumador_tabaco'
]

# Crear diccionario de nombres legibles
rename_dict = {}

for col in X_test.columns:

    renamed = False

    for var in ohe_vars:

        # Busca patrones tipo localizacion_0, localizacion_1.0, etc.
        pattern = f"^{re.escape(var)}_(.+)$"
        match = re.match(pattern, col)

        if match:
            valor = match.group(1)

            try:
                valor_num = float(valor)

                # Si existe en el mapeo
                if valor_num in mapeos[var]:
                    rename_dict[col] = f"{var}: {mapeos[var][valor_num]}"
                elif int(valor_num) in mapeos[var]:
                    rename_dict[col] = f"{var}: {mapeos[var][int(valor_num)]}"
                else:
                    rename_dict[col] = col

            except:
                rename_dict[col] = col

            renamed = True
            break

    if not renamed:
        rename_dict[col] = col

# Renombrar columnas
X_test_shap = X_test.rename(columns=rename_dict)

explainer = shap.TreeExplainer(best_rf_model_auc)

shap_values = explainer.shap_values(X_test)

shap.summary_plot(
    shap_values[:, :, 1],
    X_test_shap
)

In [ ]:


explainer = shap.TreeExplainer(best_xgb_model_kappa)

shap_values = explainer.shap_values(X_test)

shap.summary_plot(
    shap_values,
    X_test_shap
)

-----

-------------------------------------------------------------------------------------------------------------------